# Notebook C: Validation & Analysis

Repairs and validates the supply chain built in Notebook B:
entity consolidation, missing edge patches, path traceability,
port assignment comparison, and full output export.

**Reads:** `_pipeline_state_5.pkl` (from Notebook B)  
**Writes:** `_pipeline_state_6.pkl`  
**CSVs output:** `Chile_Supply_Chain_Edges.csv`, `Chile_Export_Destinations.csv`,
`Mine_Optimal_Port_Assignments.csv`, `Mine_Port_Distance_Matrix.csv`,
`Port_Distance_Comparison.csv`, `Port_Comparison_Chart.png`

**Depends on:** Notebook B


In [1]:
import sys
import os

# Find the Chile project root (contains scripts/ and inputs/) by walking up from cwd.
# Works whether you start Jupyter from Chile/ or from pipeline/.
def _find_chile_root():
    d = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(d, "scripts")) and os.path.isdir(os.path.join(d, "inputs")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError(
        "Could not find Chile project root (expected to contain scripts/ and inputs/)."

    )

_utils_dir = os.path.join(_find_chile_root(), "scripts")
if _utils_dir not in sys.path:
    sys.path.insert(0, _utils_dir)

from pipeline_utils import *
from pipeline_constants import *  # PRODUCT_TYPE_OVERRIDE, ENTITY_MERGE_MAP, MISSING_S2P, …

state            = load_state(5)
inv, links, comm_col, idle_mines, edges, ports_df = unpack_state(state)
common_cols      = state['common_cols']
smelter_inv_map  = state['smelter_inv_map']
export_df        = state['export_df']
PORT_PRODUCT_MAP = state['PORT_PRODUCT_MAP']
inv_path         = state.get('inv_path',   os.path.join(DIR_INTERMED, 'Chile_Minerals_Inventory.csv'))
links_path       = state.get('links_path', os.path.join(DIR_INTERMED, 'Chile_Mine_Plant_Links.csv'))
cu_total         = state.get('cu_total',   inv['COCHILCO_CU_2024_KMT'].sum())
n_idle_links     = state.get('n_idle_links', 0)

print(f"Loaded state from Notebook B: {len(inv)} inv rows, {len(links)} link rows, {len(edges)} edges")


Loaded state from Notebook B: 461 inv rows, 1109 link rows, 2419 edges


## 5: Cleanup & Validation

Smelter name standardisation, edge fixes, deduplication, and production validation.

In [2]:

# 5A. Smelter name standardization

section_header("5A. SMELTER NAME STANDARDIZATION")

renamed_count = 0
for canonical, inv_name in SMELTER_NAME_MAP.items():
    exists = inv["FACILITY_NAME"].eq(inv_name).any()
    print(f"  {canonical:<35} -> {inv_name:<55} {'OK' if exists else 'NOT FOUND'}")
    if canonical != inv_name:
        for col in ["FROM_NAME", "TO_NAME"]:
            mask = edges[col] == canonical
            if mask.any():
                edges.loc[mask, col] = inv_name
                renamed_count += mask.sum()
print(f"\nTotal renames: {renamed_count}")

# Handle Las Ventanas / Ventanas ambiguity
las_v   = inv[inv["FACILITY_NAME"].str.contains("Las Ventanas", case=False, na=False)]
v_plain = inv[inv["FACILITY_NAME"].str.contains("Ventanas refinery", case=False, na=False) &
              ~inv["FACILITY_NAME"].str.contains("Las Ventanas", case=False, na=False)]
if len(las_v) > 0 and len(v_plain) > 0:
    print(f"\n  Note: Both '{las_v.iloc[0]['FACILITY_NAME']}' AND '{v_plain.iloc[0]['FACILITY_NAME']}' exist.")

# 5A.2 Entity consolidation for duplicate smelter entries
#
# Several smelters appear multiple times in the inventory under slightly
# different names or as commodity-specific rows for the same physical
# facility. Edges referencing the variant names need to be remapped to
# the canonical name so that upstream and downstream connect properly.
#
# Affected facilities:
#   "Ventanas refinery and smelter" (4 rows, idx 456-459, Gold/Se/Ag/H2SO4)
#       -> canonical: "Las Ventanas refinery and smelter" (idx 340, Copper)
#   "Chagres smelter" (idx 283, Sulfuric Acid, fully disconnected)
#       -> canonical: "Chagres smelter (anodes and blister)" (idx 284, Copper)

section_header("5A.2: SMELTER ENTITY CONSOLIDATION")

# ENTITY_MERGE_MAP is defined in pipeline_constants.py (imported above).
# Do NOT redefine here — edit pipeline_constants.py to change consolidation rules.

_entity_renames = 0
for variant, canonical in ENTITY_MERGE_MAP.items():
    # Verify canonical exists in inventory
    if not inv["FACILITY_NAME"].eq(canonical).any():
        print(f"  WARNING: canonical name '{canonical}' not in inventory, skipping")
        continue
    # Remap edges
    for col in ["FROM_NAME", "TO_NAME"]:
        mask = edges[col] == variant
        if mask.any():
            n = mask.sum()
            edges.loc[mask, col] = canonical
            _entity_renames += n
            print(f"  {col}: '{variant}' -> '{canonical}' ({n} edges)")
    # Remap links table
    for col in ["MINE_NAME", "PLANT_NAME"]:
        if col in links.columns:
            mask = links[col] == variant
            if mask.any():
                links.loc[mask, col] = canonical
                print(f"  links.{col}: '{variant}' -> '{canonical}' ({mask.sum()} rows)")

if _entity_renames == 0:
    print("  No variant names found in edges (already consolidated or not present)")
else:
    print(f"\n  Total edge renames: {_entity_renames}")

# 5B. Andacollo Oro mine link

section_header("5B. ANDACOLLO ORO MINE")

andacollo = inv[inv["FACILITY_NAME"].str.contains("Andacollo", case=False, na=False)]
for _, row in andacollo.iterrows():
    prod = row.get("COCHILCO_CU_2024_KMT", np.nan)
    prod_str = f"{prod:.1f} kMT" if pd.notna(prod) else ""
    print(f"  {row['FACILITY_NAME']:<45} {row['FACILITY_TYPE']:<20} {prod_str}")

existing_anda = links[links["MINE_NAME"].str.contains("Andacollo", case=False, na=False)]
if len(existing_anda) > 0:
    print(f"  Link already exists: {existing_anda.iloc[0]['MINE_NAME']} -> {existing_anda.iloc[0]['PLANT_NAME']}")
else:
    anda_mine = inv[inv["FACILITY_NAME"].str.contains("Andacollo", case=False, na=False) &
                    inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)]
    if len(anda_mine) > 0:
        mine_row = anda_mine.iloc[0]
        if pd.notna(mine_row["LATITUD"]):
            sxew_plants = inv[(inv["CHAIN_STAGE"] == "sx_ew") & inv["LATITUD"].notna()].copy()
            sxew_plants["_dist"] = sxew_plants.apply(
                lambda r: haversine_km(mine_row["LATITUD"], mine_row["LONGITUD"], r["LATITUD"], r["LONGITUD"]), axis=1)
            nearby = sxew_plants[sxew_plants["_dist"] < 50].sort_values("_dist")
            if len(nearby) > 0:
                target = nearby.iloc[0]
                new_link = {"MINE_NAME": mine_row["FACILITY_NAME"], "PLANT_NAME": target["FACILITY_NAME"],
                            "MINE_LAT": mine_row["LATITUD"], "MINE_LON": mine_row["LONGITUD"],
                            "PLANT_LAT": target["LATITUD"], "PLANT_LON": target["LONGITUD"],
                            "SHARED_COMMODITIES": "Copper", "DISTANCE_KM": round(target["_dist"], 1),
                            "PRODUCT_FORM": "cathode_sxew"}
                links = pd.concat([links, pd.DataFrame([new_link])], ignore_index=True)
                edges = pd.concat([edges, pd.DataFrame([{
                    "FROM_NAME": mine_row["FACILITY_NAME"], "FROM_TYPE": "mine",
                    "FROM_LAT": mine_row["LATITUD"], "FROM_LON": mine_row["LONGITUD"],
                    "TO_NAME": target["FACILITY_NAME"], "TO_TYPE": "plant",
                    "TO_LAT": target["LATITUD"], "TO_LON": target["LONGITUD"],
                    "EDGE_TYPE": "mine_to_plant", "PRODUCT_FORM": "cathode_sxew",
                    "COMMODITIES": "Copper", "DISTANCE_KM": round(target["_dist"], 1),
                }])], ignore_index=True)
                print(f"  -> Created link + edge: {mine_row['FACILITY_NAME']} -> {target['FACILITY_NAME']}")

# 5B.2: Missing smelter-to-port edges
#
# Root cause: smelter_inv_map in Part 2 resolved canonical smelter names to
# SHORT inventory names via keyword search. The LINKS table, however, records
# the LONG facility name as found in the source data. This creates a split:
# mine-to-plant edges land on the long name, while smelter-to-port edges were
# built from the short search-matched name. The three smelters below ended up
# with m2p edges on their long names but zero s2p edges.
#
# Fix: add s2p edges from the long names using the SMELTERS export_port config.

section_header("5B.2: MISSING SMELTER-TO-PORT EDGES")

# MISSING_S2P lists smelters that have mine-to-plant edges under their long
# inventory names but no smelter-to-port edges. Coordinates must match
# the SMELTERS entries in pipeline_utils.py exactly.

_added_s2p = 0
for entry in MISSING_S2P:
    existing = edges[
        (edges["EDGE_TYPE"] == "smelter_to_port") &
        (edges["FROM_NAME"] == entry["from_name"])
    ]
    if len(existing) > 0:
        print(f"  {entry['from_name'][:55]} already has {len(existing)} s2p edges, skipping")
        continue
    for port_name in entry["ports"]:
        port = next((p for p in PORTS if p["name"] == port_name), None)
        if not port:
            print(f"  WARNING: port '{port_name}' not in PORTS list")
            continue
        new_edge = {
            "FROM_NAME": entry["from_name"], "FROM_TYPE": "smelter",
            "FROM_LAT": entry["lat"], "FROM_LON": entry["lon"],
            "TO_NAME": port["name"], "TO_TYPE": "port",
            "TO_LAT": port["lat"], "TO_LON": port["lon"],
            "EDGE_TYPE": "smelter_to_port",
            "PRODUCT_FORM": entry["product"],
            "COMMODITIES": "Copper",
            "DISTANCE_KM": round(haversine_km(entry["lat"], entry["lon"], port["lat"], port["lon"]), 1),
        }
        edges = pd.concat([edges, pd.DataFrame([new_edge])], ignore_index=True)
        _added_s2p += 1
        print(f"  Added: {entry['from_name'][:50]} -> {port_name}")

print(f"\n  Total new smelter-to-port edges: {_added_s2p}")

# 5B.3: Caserones mine-to-plant link
#
# Caserones (124.6 kMT) had no mine-to-plant edge, leaving it disconnected in
# path traceability. The mine has its own onsite concentrator. Try to match by
# name first; fall back to nearest concentrator within 50 km; if no concentrator
# is found, add a direct mine-to-concentrate-port edge.

section_header("5B.3: CASERONES MINE-TO-PLANT LINK")

existing_caserones = links[links["MINE_NAME"].str.contains("Caserones", case=False, na=False)]
if len(existing_caserones) > 0:
    print(f"  Link already exists: {existing_caserones.iloc[0]['MINE_NAME']} -> {existing_caserones.iloc[0]['PLANT_NAME']}")
else:
    caserones_mine = inv[
        inv["FACILITY_NAME"].str.contains("Caserones", case=False, na=False) &
        inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
    ]
    if len(caserones_mine) == 0:
        print("  No Caserones mine record found in inventory")
    else:
        mine_row = caserones_mine.iloc[0]
        print(f"  Mine: {mine_row['FACILITY_NAME']}  coords: ({mine_row['LATITUD']}, {mine_row['LONGITUD']})")
        if pd.isna(mine_row["LATITUD"]):
            print("  Mine has no coordinates — cannot auto-link")
        else:
            concentrators = inv[(inv["CHAIN_STAGE"] == "concentration") & inv["LATITUD"].notna()].copy()
            concentrators["_dist"] = concentrators.apply(
                lambda r: haversine_km(mine_row["LATITUD"], mine_row["LONGITUD"], r["LATITUD"], r["LONGITUD"]), axis=1)
            # Prefer a concentrator explicitly named Caserones
            named = concentrators[concentrators["FACILITY_NAME"].str.contains("Caserones", case=False, na=False)]
            if len(named) > 0:
                target = named.sort_values("_dist").iloc[0]
                print(f"  Matched by name: {target['FACILITY_NAME']}  ({target['_dist']:.1f} km)")
            else:
                nearby = concentrators[concentrators["_dist"] < 50].sort_values("_dist")
                target = nearby.iloc[0] if len(nearby) > 0 else None
                if target is not None:
                    print(f"  Matched by proximity: {target['FACILITY_NAME']}  ({target['_dist']:.1f} km)")

            if target is not None:
                dist_km = round(float(target["_dist"]), 1)
                new_link = {
                    "MINE_NAME": mine_row["FACILITY_NAME"], "PLANT_NAME": target["FACILITY_NAME"],
                    "MINE_LAT": mine_row["LATITUD"], "MINE_LON": mine_row["LONGITUD"],
                    "PLANT_LAT": target["LATITUD"], "PLANT_LON": target["LONGITUD"],
                    "SHARED_COMMODITIES": "Copper", "DISTANCE_KM": dist_km,
                    "PRODUCT_FORM": "concentrate",
                }
                links = pd.concat([links, pd.DataFrame([new_link])], ignore_index=True)
                edges = pd.concat([edges, pd.DataFrame([{
                    "FROM_NAME": mine_row["FACILITY_NAME"], "FROM_TYPE": "mine",
                    "FROM_LAT": mine_row["LATITUD"], "FROM_LON": mine_row["LONGITUD"],
                    "TO_NAME": target["FACILITY_NAME"], "TO_TYPE": "plant",
                    "TO_LAT": target["LATITUD"], "TO_LON": target["LONGITUD"],
                    "EDGE_TYPE": "mine_to_plant", "PRODUCT_FORM": "concentrate",
                    "COMMODITIES": "Copper", "DISTANCE_KM": dist_km,
                }])], ignore_index=True)
                print(f"  -> Added link + edge: {mine_row['FACILITY_NAME']} -> {target['FACILITY_NAME']}")
            else:
                # No concentrator found: add direct mine-to-port edge so the
                # 124.6 kMT is not silently excluded from traceability.
                port, dist = nearest_port(mine_row["LATITUD"], mine_row["LONGITUD"], "concentrate")
                if port:
                    edges = pd.concat([edges, pd.DataFrame([{
                        "FROM_NAME": mine_row["FACILITY_NAME"], "FROM_TYPE": "mine",
                        "FROM_LAT": mine_row["LATITUD"], "FROM_LON": mine_row["LONGITUD"],
                        "TO_NAME": port["name"], "TO_TYPE": "port",
                        "TO_LAT": port["lat"], "TO_LON": port["lon"],
                        "EDGE_TYPE": "concentrate_to_port", "PRODUCT_FORM": "concentrate",
                        "COMMODITIES": "Copper", "DISTANCE_KM": round(dist, 1),
                    }])], ignore_index=True)
                    print(f"  -> No concentrator found; added direct mine-to-port edge -> {port['name']}")

# 5C. Deduplication

section_header("5C. DEDUPLICATION")

before = len(edges)
edges = edges.drop_duplicates(
    subset=["FROM_NAME", "TO_NAME", "EDGE_TYPE", "COMMODITIES", "PRODUCT_FORM"], keep="first"
)
print(f"  Removed {before - len(edges)} duplicate edges ({before} -> {len(edges)})")

# 5D. Validation

section_header("5D. VALIDATION")

issues = []
cu_matched_total = inv["COCHILCO_CU_2024_KMT"].sum()
cu_count = inv["COCHILCO_CU_2024_KMT"].notna().sum()
print(f"  Cu production: {cu_count} records, {cu_matched_total:,.1f} / {cu_total:,.1f} kMT "
      f"({cu_matched_total/cu_total*100:.1f}%)")

for _mineral_name, _col_name, _unit_label in [
    ("Mo", "COCHILCO_MO_2024_MT", "MT"),
    ("Au", "COCHILCO_AU_2024_KG", "Kg"),
    ("Ag", "COCHILCO_AG_2024_KG", "Kg"),
    ("Fe", "COCHILCO_FE_2024_KMT", "kMT"),
    ("Zn", "COCHILCO_ZN_2024_MT", "MT"),
]:
    if _col_name in inv.columns and inv[_col_name].notna().any():
        _n = inv[_col_name].notna().sum()
        _total = inv[_col_name].sum()
        print(f"  {_mineral_name} production: {_n} records, {_total:,.1f} {_unit_label}")

# Path traceability: for each Cu producer, check whether a route to a port exists.
# Uses set operations instead of a nested loop — the edge table is too large for
# per-row DataFrame scans to be practical at this scale.
_port_sources = set(edges.loc[
    edges["EDGE_TYPE"].isin(["concentrate_to_port", "sxew_to_port", "smelter_to_port"]), "FROM_NAME"
])
_c2s = edges[edges["EDGE_TYPE"] == "concentrate_to_smelter"][["FROM_NAME", "TO_NAME"]]
_smelter_port_sources = set(edges.loc[edges["EDGE_TYPE"] == "smelter_to_port", "FROM_NAME"])
_via_smelter = set(_c2s.loc[_c2s["TO_NAME"].isin(_smelter_port_sources), "FROM_NAME"])
_plants_reach_port = _port_sources | _via_smelter

_m2p = edges[edges["EDGE_TYPE"] == "mine_to_plant"][["FROM_NAME", "TO_NAME"]]
_mine_reaches = (
    _m2p.groupby("FROM_NAME")["TO_NAME"]
    .apply(set)
    .reset_index()
    .rename(columns={"FROM_NAME": "MINE", "TO_NAME": "PLANTS"})
)
_mine_reaches["reaches_port"] = _mine_reaches["PLANTS"].apply(
    lambda plants: bool(plants & _plants_reach_port)
)
_connected_mines = set(_mine_reaches.loc[_mine_reaches["reaches_port"], "MINE"])

# Some SX-EW operations (e.g. Zaldivar, Tres Valles) have COCHILCO production
# attached directly to the plant record rather than a separate mine record.
# Check whether the producer appears in sxew_to_port or concentrate_to_port.
_direct_to_port = set(edges.loc[
    edges["EDGE_TYPE"].isin(["sxew_to_port", "concentrate_to_port"]), "FROM_NAME"
])

cu_producers = inv[(inv["COCHILCO_CU_2024_KMT"].notna()) & (inv["COCHILCO_CU_2024_KMT"] > 0)]
connected_prod, disconnected = 0.0, []

for _, mrow in cu_producers.iterrows():
    mine_name = mrow["FACILITY_NAME"]
    is_connected = (
        mine_name in _connected_mines
        # NOTE: fuzzy prefix match removed.  The original
        #   any(m.startswith(mine_name[:15]) for m in _connected_mines)
        # produced false-positives: short names (e.g. "Cerro Negro") matched
        # longer connected names (e.g. "Cerro Negro Norte") because the slice
        # is taken from the SEARCH name, not the connected name.  Exact matching
        # is sufficient; name variants should be resolved via ENTITY_MERGE_MAP.
        or mine_name in _direct_to_port  # direct sx_ew / concentrator producers
    )
    if is_connected:
        connected_prod += mrow["COCHILCO_CU_2024_KMT"]
    else:
        disconnected.append((mine_name, mrow["COCHILCO_CU_2024_KMT"]))

print(f"\n  Path traceability:")
print(f"    Mines reaching port: {cu_count - len(disconnected)} / {cu_count}")
print(f"    Production reaching ports: {connected_prod:,.1f} / {cu_matched_total:,.1f} kMT "
      f"({connected_prod/cu_matched_total*100:.1f}%)")
if disconnected:
    for name, prod in sorted(disconnected, key=lambda x: -x[1]):
        print(f"      {name:<45} {prod:>8.1f} kMT")
    issues.append(f"{len(disconnected)} mines ({sum(p for _,p in disconnected):,.1f} kMT) don't reach a port")

section_header("DIAGNOSTICS SUMMARY")

for etype in edges["EDGE_TYPE"].unique():
    n = len(edges[edges["EDGE_TYPE"] == etype])
    print(f"  {etype:<25} {n:>5} edges")

print(f"\n  Downstream coverage by commodity:")
downstream_check = edges[edges["EDGE_TYPE"] != "mine_to_plant"]
if "COMMODITIES" in downstream_check.columns:
    for comm, count in downstream_check["COMMODITIES"].value_counts().items():
        print(f"    {comm:<20} {count:>5} downstream edges")

print(f"\n  Smelter connectivity:")
for name in sorted(inv[inv["CHAIN_STAGE"] == "smelting"]["FACILITY_NAME"].unique()):
    m2p = len(edges[(edges["EDGE_TYPE"] == "mine_to_plant") & (edges["TO_NAME"] == name)])
    s2p = len(edges[(edges["EDGE_TYPE"] == "smelter_to_port") & (edges["FROM_NAME"] == name)])
    conn = f"m2p={m2p}, s2p={s2p}" if (m2p + s2p) > 0 else "DISCONNECTED"
    print(f"    {name:<60} [{conn}]")

print(f"\nNote: {len(idle_mines)} idle mines retained in inventory, {n_idle_links} phantom links removed.")

section_header("5F. SAVE")

edges = edges.copy()  # avoid SettingWithCopyWarning on in-place sort
edges.sort_values(["EDGE_TYPE", "FROM_NAME", "TO_NAME"], inplace=True)
edges.reset_index(drop=True, inplace=True)

for et, count in edges["EDGE_TYPE"].value_counts().sort_index().items():
    print(f"  {et:<25} {count:>5}")
print(f"  {'TOTAL':<25} {len(edges):>5}")

if issues:
    print(f"\nRemaining issues:")
    for i, iss in enumerate(issues, 1):
        print(f"  {i}. {iss}")

ds = edges[edges["EDGE_TYPE"] != "mine_to_plant"]
export_df.to_csv(os.path.join(DIR_INTERMED, "Chile_Export_Destinations.csv"), index=False)
edges.to_csv(os.path.join(DIR_INTERMED, "Chile_Supply_Chain_Edges.csv"), index=False)
inv.to_csv(inv_path, index=False)
links.to_csv(links_path, index=False)
ds.to_csv(os.path.join(DIR_INTERMED, "Chile_Downstream_Links.csv"), index=False)
ports_df.to_csv(os.path.join(DIR_INTERMED, "Chile_Ports.csv"), index=False)

print(f"\nSaved:")
for name, n in [("Chile_Minerals_Inventory.csv", len(inv)), ("Chile_Mine_Plant_Links.csv", len(links)),
                ("Chile_Supply_Chain_Edges.csv", len(edges)), ("Chile_Downstream_Links.csv", len(ds)),
                ("Chile_Export_Destinations.csv", len(export_df)), ("Chile_Ports.csv", len(ports_df))]:
    print(f"  {name:<35} ({n} records)")


5A. SMELTER NAME STANDARDIZATION
  Chuquicamata smelter                -> Chuquicamata SX-EW plant (oxide) and smelter            OK
  Potrerillos smelter                 -> Potrerillos SX-EW refinery and smelter                  OK
  Caletones smelter                   -> Caletones smelter (anodes). refinery (fire-refined ingots), and SX-EW plant OK
  Altonorte smelter                   -> Altonorte smelter                                       OK
  Paipote smelter (H.V. Lira)         -> Hernán Videla Lira smelter (anodes and blister)         OK
  Chagres smelter                     -> Chagres smelter (anodes and blister)                    OK

Total renames: 6

  Note: Both 'Las Ventanas refinery and smelter' AND 'Ventanas refinery and smelter' exist.

5A.2: SMELTER ENTITY CONSOLIDATION
  No variant names found in edges (already consolidated or not present)

5B. ANDACOLLO ORO MINE
  Andacollo Oro                                 Mine (active)        
  Carmen de Andacollo            

## 6: Distance-Based Port Assignment Analysis

Haversine distances between mines and ports, actual vs optimal port assignment comparison.

In [3]:

section_header("6. DISTANCE-BASED PORT ASSIGNMENT ANALYSIS")

mines = inv[
    (inv["CHAIN_STAGE"] == "extraction") & inv["LATITUD"].notna() &
    inv["LONGITUD"].notna() & (inv["COCHILCO_CU_2024_KMT"].notna()) &
    (inv["COCHILCO_CU_2024_KMT"] > 0)
].copy()

# Also include SX-EW facilities with production matched directly (e.g. Zaldivar,
# Tres Valles) — pure SX-EW operations with no separate mine inventory record.
sxew_producers = inv[
    (inv["CHAIN_STAGE"] == "sx_ew") & inv["LATITUD"].notna() &
    inv["LONGITUD"].notna() & (inv["COCHILCO_CU_2024_KMT"].notna()) &
    (inv["COCHILCO_CU_2024_KMT"] > 0) &
    ~inv["FACILITY_NAME"].isin(mines["FACILITY_NAME"])  # not already in mines
].copy()

if len(sxew_producers) > 0:
    print(f"  Adding {len(sxew_producers)} SX-EW producers with direct production data:")
    for _, r in sxew_producers.iterrows():
        print(f"    {r['FACILITY_NAME']:<45} {r['COCHILCO_CU_2024_KMT']:.1f} kMT")
    mines = pd.concat([mines, sxew_producers], ignore_index=True)

print(f"Mines with production data: {len(mines)}")
print(f"Ports: {len(ports_df)}")

# Product type classification
#
# Determines whether each mine's copper output leaves Chile as concentrate,
# cathode (SX-EW or fire-refined), or blister.
#
# Priority: (1) hardcoded override, (2) comm_col on the mine record,
# (3) nearby non-mine facilities in inventory, (4) mode of PRODUCT_FORM in links.
#
# Key corrections vs. the prior run:
# - Codelco mines (Chuquicamata, El Teniente, Radomiro Tomic, etc.) are marked
#   'cathode' because their concentrate is processed onsite/nearby and the final
#   export product is refined cathode, routed via CODELCO_CATHODE_ROUTING.
# - Pure SX-EW mines (Zaldívar, Tres Valles, Antucoya, etc.) are 'cathode'.
# - Mines that ship raw concentrate for third-party smelting remain 'concentrate'.

# PRODUCT_TYPE_OVERRIDE is defined in pipeline_constants.py (imported above).
# It now includes 'Rio Blanco': 'cathode' to cover División Andina's primary
# ore body, which was missing from the original notebook definition.
# Do NOT redefine here — edit pipeline_constants.py to change routing rules.

def classify_mine_product(mine_name):
    # 1. Hardcoded overrides (highest priority)
    for key, ptype in PRODUCT_TYPE_OVERRIDE.items():
        if key.lower() in mine_name.lower():
            return ptype

    # 2. Check comm_col on the mine record directly
    mine_rows = inv[
        inv["FACILITY_NAME"].str.contains(mine_name, case=False, na=False, regex=False) &
        inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
    ]
    if len(mine_rows) > 0 and comm_col in mine_rows.columns:
        comm_str = str(mine_rows.iloc[0].get(comm_col, "")).lower()
        if "cathode" in comm_str or "sxew" in comm_str:
            return "cathode"
        if "concentrate" in comm_str:
            return "concentrate"
        if "blister" in comm_str:
            return "blister"

    # 3. Nearby non-mine facilities in inventory
    tokens = mine_name.split()
    search_terms = [" ".join(tokens[:2])] if len(tokens) >= 2 and tokens[0] in ("El", "Los", "Las", "La") else []
    search_terms.append(tokens[0] if len(tokens[0]) >= 5 else mine_name)
    for term in search_terms:
        candidates = inv[
            inv["FACILITY_NAME"].str.contains(term, case=False, na=False, regex=False) &
            ~inv["FACILITY_TYPE"].str.contains("Mine|Prospect", case=False, na=False)
        ]
        if len(candidates) > 0:
            stages = set(candidates["CHAIN_STAGE"].dropna())
            if stages & {"concentration", "smelting", "processing"}: return "concentrate"
            if "sx_ew" in stages: return "cathode"

    # 4. Fallback: mode of PRODUCT_FORM in links table
    mine_links = links[links["MINE_NAME"] == mine_name]
    if len(mine_links) > 0:
        mode = mine_links["PRODUCT_FORM"].value_counts().index[0]
        return {"concentrate": "concentrate", "cathode_sxew": "cathode",
                "cathode_er": "cathode", "blister": "blister"}.get(mode, "unknown")
    return "unknown"

mines["product_type"] = mines["FACILITY_NAME"].apply(classify_mine_product)
print(f"\nMines by product type:\n{mines['product_type'].value_counts().to_string()}")

# Warn on unclassified mines so no production is silently excluded
unknown_mines = mines[mines["product_type"] == "unknown"]
if len(unknown_mines) > 0:
    unknown_prod  = unknown_mines["COCHILCO_CU_2024_KMT"].sum()
    total_prod    = mines["COCHILCO_CU_2024_KMT"].sum()
    print(f"\nWARNING: {len(unknown_mines)} mines ({unknown_prod:.1f} kMT, "
          f"{unknown_prod/total_prod*100:.1f}% of total) unclassified and excluded from port shares:")
    for _, r in unknown_mines.iterrows():
        print(f"  {r['FACILITY_NAME']:<45} {r['COCHILCO_CU_2024_KMT']:>8.1f} kMT")

# Vectorised distance matrix
# Full mine x port distance matrix computed in one numpy broadcasting pass.

lat_m = np.radians(mines["LATITUD"].values)          # shape (M,)
lon_m = np.radians(mines["LONGITUD"].values)
lat_p = np.radians(ports_df["lat"].values)           # shape (P,)
lon_p = np.radians(ports_df["lon"].values)

dlat = lat_m[:, None] - lat_p[None, :]               # (M, P)
dlon = lon_m[:, None] - lon_p[None, :]
a    = (np.sin(dlat / 2) ** 2
        + np.cos(lat_m[:, None]) * np.cos(lat_p[None, :])
        * np.sin(dlon / 2) ** 2)
distances = 6371.0 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

distance_df = pd.DataFrame(
    distances,
    index=mines["FACILITY_NAME"].values,
    columns=ports_df["name"].values,
)

# Port assignment: dedicated overrides > Codelco routing > nearest

def assign_port(mine_row, dist_row):
    """Assign port using: dedicated override > Codelco routing > nearest distance."""
    mine_name = mine_row["FACILITY_NAME"].lower()
    product   = mine_row["product_type"]

    # 1. Dedicated port override (contractual)
    for key, port_name in DEDICATED_PORT.items():
        if key.lower() in mine_name and port_name in dist_row.index:
            return port_name, dist_row[port_name]

    # 2. Codelco cathode consolidation
    if product == "cathode":
        for key, port_name in CODELCO_CATHODE_ROUTING.items():
            if key.lower() in mine_name and port_name in dist_row.index:
                return port_name, dist_row[port_name]

    # 3. Nearest port by straight-line distance (default)
    return dist_row.idxmin(), dist_row.min()

assigned_ports, assigned_dists = [], []
for i, (_, mine_row) in enumerate(mines.iterrows()):
    port_name, port_dist = assign_port(mine_row, distance_df.iloc[i])
    assigned_ports.append(port_name)
    assigned_dists.append(port_dist)

mines["nearest_port"] = assigned_ports
mines["distance_km"]  = assigned_dists

print(f"\nTop 10 mines by production with assigned port:")
for _, mine in mines.nlargest(10, "COCHILCO_CU_2024_KMT").iterrows():
    mine_lower = mine["FACILITY_NAME"].lower()
    is_override = (
        any(k.lower() in mine_lower for k in DEDICATED_PORT) or
        (mine["product_type"] == "cathode" and
         any(k.lower() in mine_lower for k in CODELCO_CATHODE_ROUTING))
    )
    marker = " [override]" if is_override else ""
    print(f"  {mine['FACILITY_NAME']:<40} -> {mine['nearest_port']:<25} "
          f"({mine['distance_km']:.0f} km, {mine['COCHILCO_CU_2024_KMT']:.1f} kMT){marker}")

# Simulated port shares by product type

simulated_shares = {}
for product_type in ["concentrate", "cathode", "blister"]:
    product_mines = mines[mines["product_type"] == product_type]
    if len(product_mines) == 0: continue
    total_production = product_mines["COCHILCO_CU_2024_KMT"].sum()
    port_production  = product_mines.groupby("nearest_port")["COCHILCO_CU_2024_KMT"].sum()
    port_shares = (port_production / total_production).to_dict()
    simulated_shares[product_type] = {k: v for k, v in port_shares.items() if v >= 0.01}
    print(f"\n{product_type.upper()} (modelled):")
    for port, share in sorted(simulated_shares[product_type].items(), key=lambda x: -x[1]):
        print(f"  {port:<30} {share*100:>6.1f}%")

# Compare to actual Aduanas port shares
# Expected file: Chile_Port_Shares_Aduanas.csv
# Required columns: PORT, PRODUCT (concentrate/cathode/blister), FOB_SHARE (0-1)

port_shares_path = os.path.join(DIR_PRELIM, "Chile_Port_Shares_Aduanas.csv")
if os.path.exists(port_shares_path):
    actual_shares_df = pd.read_csv(port_shares_path)
    # Use ADUANAS_PORT_ALIAS from pipeline_constants.py for port name normalisation.
    # Source: Aduanas all-caps Spanish -> mixed-case English used in PORTS list.
    # Risk: the alias dict may not cover all codes in newer Salidas files;
    # review ADUANAS_PORT_ALIAS in pipeline_constants.py if a port is unmapped.
    actual_shares_df["PORT"] = actual_shares_df["PORT"].replace(ADUANAS_PORT_ALIAS)

    actual_shares = {}
    for product in ["concentrate", "cathode", "blister"]:
        product_data = actual_shares_df[actual_shares_df["PRODUCT"] == product]
        if len(product_data) == 0: continue
        port_share_dict = product_data.groupby("PORT")["FOB_SHARE"].sum().to_dict()
        actual_shares[product] = {k: v for k, v in port_share_dict.items() if v >= 0.01}
        print(f"\n{product.upper()} (actual, Aduanas):")
        for port, share in sorted(actual_shares[product].items(), key=lambda x: -x[1]):
            print(f"  {port:<30} {share*100:>6.1f}%")

    section_header("COMPARISON: ACTUAL vs MODELLED")
    comparison_results = []
    for product_type in ["concentrate", "cathode", "blister"]:
        if product_type not in simulated_shares: continue
        actual    = actual_shares.get(product_type, {})
        simulated = simulated_shares[product_type]
        all_ports = set(actual.keys()) | set(simulated.keys())
        print(f"\n{product_type.upper()}\n" + "-" * 75)
        for port in sorted(all_ports):
            a, s = actual.get(port, 0), simulated.get(port, 0)
            diff = s - a
            indicator = "=" if abs(diff) < 0.02 else ("^ (would gain)" if diff > 0 else "v (would lose)")
            comparison_results.append({"product": product_type, "port": port,
                                       "actual_share": a, "simulated_share": s, "difference": diff})
            print(f"  {port:<30} Actual: {a*100:5.1f}%  |  Modelled: {s*100:5.1f}%  |  D {diff*100:+6.1f}% {indicator}")

    comparison_df = pd.DataFrame(comparison_results)

    section_header("SUMMARY STATISTICS")
    for product_type in ["concentrate", "cathode", "blister"]:
        pc = comparison_df[comparison_df["product"] == product_type]
        if len(pc) == 0: continue
        print(f"\n{product_type.upper()}:")
        print(f"  Mean Absolute Difference: {pc['difference'].abs().mean()*100:.1f}%")
        print(f"  Total Volume Mismatch:    {pc['difference'].abs().sum()/2*100:.1f}%")

    # Structural limitation note
    # The mismatch figures above (~20-30% mean absolute deviation) reflect a
    # known structural limitation: this model assigns each mine to a SINGLE port
    # using a priority order (override → dedicated → nearest-by-distance). Real
    # operators split shipments across multiple ports depending on vessel
    # availability, contract terms, and seasonal capacity constraints. A
    # single-port routing model systematically over-assigns to the
    # geographically nearest terminal and under-assigns to secondary ports.
    # These figures are NOT errors in the pipeline — they document the gap
    # between a simplified routing model and observed Aduanas data.
    print("\nNOTE: Mismatch figures reflect single-port routing assumption.")
    print("  Real operators split shipments across multiple ports; the model cannot")
    print("  capture this without mine-level shipment data. See pipeline documentation.")
    comparison_df.to_csv(os.path.join(DIR_PRELIM, "Port_Distance_Comparison.csv"), index=False)
    mines[["FACILITY_NAME", "COCHILCO_CU_2024_KMT", "product_type",
           "nearest_port", "distance_km"]].to_csv(
        os.path.join(DIR_PRELIM, "Mine_Optimal_Port_Assignments.csv"), index=False)
    distance_df.to_csv(os.path.join(DIR_PRELIM, "Mine_Port_Distance_Matrix.csv"))

    # Visualisation
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle("Port Share Comparison: Actual vs Modelled (overrides + distance)",
                 fontsize=16, fontweight="bold")
    colors = {"actual": "#FF6B6B", "simulated": "#4ECDC4"}

    for idx, product in enumerate(["concentrate", "cathode", "blister"]):
        ax = axes[idx]
        if product not in simulated_shares:
            ax.text(0.5, 0.5, f"No {product} data", ha="center", va="center", transform=ax.transAxes)
            continue
        pd_data = comparison_df[comparison_df["product"] == product].copy()
        pd_data = pd_data.sort_values("actual_share", ascending=False)
        pd_data = pd_data[(pd_data["actual_share"] >= 0.01) | (pd_data["simulated_share"] >= 0.01)].head(8)
        if len(pd_data) == 0: continue
        x     = np.arange(len(pd_data))
        width = 0.35
        ax.bar(x - width/2, pd_data["actual_share"]    * 100, width, label="Actual",   color=colors["actual"],    alpha=0.8)
        ax.bar(x + width/2, pd_data["simulated_share"] * 100, width, label="Modelled", color=colors["simulated"], alpha=0.8)
        ax.set_xlabel("Port"); ax.set_ylabel("Share (%)")
        ax.set_title(f"{product.upper()}", fontsize=13, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(pd_data["port"], rotation=45, ha="right", fontsize=9)
        ax.legend(); ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(DIR_PRELIM, "Port_Comparison_Chart.png"), dpi=300, bbox_inches="tight")
    print(f"\nChart saved: Port_Comparison_Chart.png")
    plt.show()

else:
    print("\nWarning: Chile_Port_Shares_Aduanas.csv not found — only modelled shares computed.")
    print(f"  Expected path: {port_shares_path}")
    print("  Required columns: PORT, PRODUCT (concentrate/cathode/blister), FOB_SHARE (0-1 decimal)")
    comparison_df = pd.DataFrame()
    mines[["FACILITY_NAME", "COCHILCO_CU_2024_KMT", "product_type",
           "nearest_port", "distance_km"]].to_csv(
        os.path.join(DIR_PRELIM, "Mine_Optimal_Port_Assignments.csv"), index=False)
    distance_df.to_csv(os.path.join(DIR_PRELIM, "Mine_Port_Distance_Matrix.csv"))
    print("Saved: Mine_Optimal_Port_Assignments.csv, Mine_Port_Distance_Matrix.csv")



6. DISTANCE-BASED PORT ASSIGNMENT ANALYSIS
  Adding 2 SX-EW producers with direct production data:
    Tres Valles SX-EW plant                       1.5 kMT
    Zaldívar SX-EW plant                          82.8 kMT
Mines with production data: 31
Ports: 13

Mines by product type:
product_type
cathode        18
concentrate    13

Top 10 mines by production with assigned port:
  Escondida                                -> Coloso                    (153 km, 1277.5 kMT) [override]
  Collahuasi                               -> Patache                   (159 km, 558.6 kMT)
  El Teniente                              -> San Antonio               (131 km, 356.4 kMT) [override]
  Los Pelambres                            -> Los Vilos                 (98 km, 331.2 kMT) [override]
  Chuquicamata                             -> Angamos                   (180 km, 289.0 kMT) [override]
  Radomiro Tomic                           -> Angamos                   (185 km, 270.5 kMT) [override]
  Spence      

## 7: Pipeline Summary & Integrity Checks

In [4]:
section_header("1. INVENTORY SUMMARY")

print(f"Total facilities:  {len(inv)}")
print(f"Idle mines:        {len(idle_mines)}")
print(f"Idle links removed:{n_idle_links}")

print("\nFacility types:")
for ftype, n in inv["FACILITY_TYPE"].value_counts().items():
    print(f"  {ftype:<40} {n:>5}")

print("\nCommodity coverage (facilities with each commodity):")
comm_counts = {}
for val in inv[comm_col].dropna():
    for c in [x.strip() for x in str(val).split(",") if x.strip()]:
        comm_counts[c] = comm_counts.get(c, 0) + 1
for comm, n in sorted(comm_counts.items(), key=lambda x: -x[1]):
    print(f"  {comm:<25} {n:>5} facilities")

# Integrity label
_n_no_coords = inv["LATITUD"].isna().sum()
_n_no_comm   = inv[comm_col].isna().sum()
if _n_no_coords == 0 and _n_no_comm == 0:
    print("\nCHECK 1 (Inventory summary): PASS")
elif _n_no_coords < 30 and _n_no_comm < 30:
    print(f"\nCHECK 1 (Inventory summary): WARN  (missing coords: {_n_no_coords}, missing commodity: {_n_no_comm})")
else:
    print(f"\nCHECK 1 (Inventory summary): FAIL  (missing coords: {_n_no_coords}, missing commodity: {_n_no_comm})")



1. INVENTORY SUMMARY
Total facilities:  461
Idle mines:        80
Idle links removed:0

Facility types:
  Processing Plant                           123
  Prospect/Project                           113
  Mine (idle)                                 80
  Mine (active)                               74
  SX-EW Plant                                 23
  Concentrator                                22
  Smelter                                     14
  Refinery                                     4
  Grinding Plant                               4
  Pellet Plant                                 2
  Steel Plant                                  2

Commodity coverage (facilities with each commodity):
  Copper                      193 facilities
  Gold                        103 facilities
  Silver                       63 facilities
  Molybdenum                   44 facilities
  Iron                         26 facilities
  Nitrate                      21 facilities
  Iodine                       2

In [5]:
section_header("2. SUPPLY CHAIN EDGES")

print(f"Total edges: {len(edges)}")

print("\nEdge types:")
for et, n in edges["EDGE_TYPE"].value_counts().sort_index().items():
    print(f"  {et:<35} {n:>5}")

print("\nCommodities in edges:")
for comm, n in edges["COMMODITIES"].value_counts().items():
    print(f"  {comm:<25} {n:>5} edges")

print("\nProduct forms:")
for pf, n in edges["PRODUCT_FORM"].value_counts().items():
    print(f"  {pf:<25} {n:>5} edges")

# Integrity label
_n_edges = len(edges)
_n_mine2plant = (edges["EDGE_TYPE"] == "mine_to_plant").sum()
if _n_edges > 500 and _n_mine2plant > 100:
    print(f"\nCHECK 2 (Supply chain edges): PASS  ({_n_edges} total edges, {_n_mine2plant} mine_to_plant)")
elif _n_edges > 0:
    print(f"\nCHECK 2 (Supply chain edges): WARN  ({_n_edges} edges, {_n_mine2plant} mine_to_plant — lower than expected)")
else:
    print("\nCHECK 2 (Supply chain edges): FAIL  (no edges in table)")



2. SUPPLY CHAIN EDGES
Total edges: 2425

Edge types:
  boron_to_port                           3
  concentrate_to_port                     6
  concentrate_to_smelter                  7
  gold_to_airport                        10
  iodine_to_port                          6
  iron_to_port                            8
  lithium_to_port                         3
  mine_to_mo_plant                       12
  mine_to_plant                        1157
  mine_to_smelter                        78
  mo_to_port                              4
  mo_to_re_plant                          4
  nitrate_to_port                         7
  port_to_country                      1051
  processing_to_port                     19
  rhenium_to_port                         1
  silver_to_airport                       7
  smelter_to_port                        15
  sxew_to_port                           25
  zinc_to_port                            2

Commodities in edges:
  Copper                     1320 edges
  G

In [6]:
section_header("3. COPPER PRODUCTION COVERAGE")

cu_col = "COCHILCO_CU_2024_KMT"
cu_inv = inv[inv[cu_col].notna() & (inv[cu_col] > 0)].copy()
cu_inv = cu_inv.sort_values(cu_col, ascending=False)

print(f"Active Cu producers in inventory: {len(cu_inv)}")
print(f"Total Cu 2024 (kMT):              {cu_inv[cu_col].sum():.1f}")
print(f"Cu total from state:              {cu_total:.1f}")

print("\nTop 15 Cu producers:")
for _, row in cu_inv.head(15).iterrows():
    prod = row[cu_col]
    share = prod / cu_inv[cu_col].sum() * 100
    print(f"  {row['FACILITY_NAME']:<45} {prod:>7.1f} kMT  ({share:>5.1f}%)")

# Integrity label
_cu_sum = cu_inv["COCHILCO_CU_2024_KMT"].sum() if len(cu_inv) > 0 else 0
_coverage = _cu_sum / cu_total * 100 if cu_total > 0 else 0
if _coverage >= 90:
    print(f"\nCHECK 3 (Cu production coverage): PASS  ({_coverage:.1f}% of national total matched)")
elif _coverage >= 70:
    print(f"\nCHECK 3 (Cu production coverage): WARN  ({_coverage:.1f}% matched — below 90% threshold)")
else:
    print(f"\nCHECK 3 (Cu production coverage): FAIL  ({_coverage:.1f}% matched — significant gap)")



3. COPPER PRODUCTION COVERAGE
Active Cu producers in inventory: 31
Total Cu 2024 (kMT):              5281.6
Cu total from state:              5506.0

Top 15 Cu producers:
  Escondida                                      1277.5 kMT  ( 24.2%)
  Collahuasi                                      558.6 kMT  ( 10.6%)
  El Teniente                                     356.4 kMT  (  6.7%)
  Los Pelambres                                   331.2 kMT  (  6.3%)
  Chuquicamata                                    289.0 kMT  (  5.5%)
  Radomiro Tomic                                  270.5 kMT  (  5.1%)
  Spence                                          255.6 kMT  (  4.8%)
  Quebrada Blanca                                 207.8 kMT  (  3.9%)
  Andina                                          181.6 kMT  (  3.4%)
  Los Bronces                                     172.4 kMT  (  3.3%)
  Sierra Gorda                                    154.6 kMT  (  2.9%)
  Candelaria                                      142.1 kM

In [7]:
section_header("4. CONNECTIVITY CHECKS")

# Graph reachability: every copper mine should reach at least one port
# This is a proper BFS over the full edge table, not just a membership check
# in the mine_to_plant FROM_NAME list. A mine passes iff there exists at least
# one directed path mine -> ... -> port in the edge DAG.

_port_names = {p["name"] for p in PORTS}

# Build adjacency dict from the full edge table (directed graph)
from collections import deque, defaultdict
_adj = defaultdict(set)
for _, row in edges.iterrows():
    _adj[row["FROM_NAME"]].add(row["TO_NAME"])

def _reaches_port(start_name: str) -> bool:
    """BFS: return True if start_name can reach any known port node."""
    visited = {start_name}
    queue = deque([start_name])
    while queue:
        node = queue.popleft()
        if node in _port_names:
            return True
        for nbr in _adj.get(node, set()):
            if nbr not in visited:
                visited.add(nbr)
                queue.append(nbr)
    return False

# Active mines in inventory
active_mines = inv[inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
                   & ~inv["FACILITY_TYPE"].str.contains("idle", case=False, na=False)]

# Mines referenced in edges (simple membership check for connectivity)
mine_names_in_edges = set(edges[edges["EDGE_TYPE"] == "mine_to_plant"]["FROM_NAME"].unique())

connected = active_mines[active_mines["FACILITY_NAME"].isin(mine_names_in_edges)]
unconnected = active_mines[~active_mines["FACILITY_NAME"].isin(mine_names_in_edges)]

print(f"Active mines in inventory:    {len(active_mines)}")
print(f"Mines with mine_to_plant edge:{len(connected)}")
print(f"Mines with no edges:          {len(unconnected)}")

# BFS reachability for copper producers specifically
cu_mines = active_mines[
    active_mines["COCHILCO_CU_2024_KMT"].notna() &
    (active_mines["COCHILCO_CU_2024_KMT"] > 0)
]
port_reachable = [nm for nm in cu_mines["FACILITY_NAME"] if _reaches_port(nm)]
port_unreachable = [nm for nm in cu_mines["FACILITY_NAME"] if not _reaches_port(nm)]

print(f"\nCopper producers with graph-reachable port: {len(port_reachable)} / {len(cu_mines)}")
if port_unreachable:
    print("WARN: copper producers that cannot reach any port via edge graph:")
    for nm in port_unreachable:
        cu = cu_mines.loc[cu_mines["FACILITY_NAME"] == nm, "COCHILCO_CU_2024_KMT"].values[0]
        print(f"  {nm:<45} {cu:.1f} kMT")
else:
    print("  All copper producers reach at least one port. PASS")

if len(unconnected) > 0:
    print("\nUnconnected mines (no mine_to_plant edge):")
    for _, row in unconnected.iterrows():
        cu = row.get("COCHILCO_CU_2024_KMT", float("nan"))
        comm = row.get(comm_col, "")
        print(f"  {row['FACILITY_NAME']:<45} Cu={cu}  {comm}")

# Integrity label
if len(port_unreachable) == 0 and len(unconnected) == 0:
    print("\nCHECK 4 (Connectivity): PASS")
elif len(port_unreachable) == 0:
    print(f"\nCHECK 4 (Connectivity): WARN  ({len(unconnected)} active mines have no edges, but all Cu producers reach a port)")
else:
    print(f"\nCHECK 4 (Connectivity): FAIL  ({len(port_unreachable)} Cu producers cannot reach any port)")



4. CONNECTIVITY CHECKS
Active mines in inventory:    74
Mines with mine_to_plant edge:71
Mines with no edges:          3

Copper producers with graph-reachable port: 29 / 29
  All copper producers reach at least one port. PASS

Unconnected mines (no mine_to_plant edge):
  Andacollo Oro                                 Cu=nan  Gold
  El Sauce                                      Cu=nan  Copper
  Pampa Camarones                               Cu=nan  Copper

CHECK 4 (Connectivity): WARN  (3 active mines have no edges, but all Cu producers reach a port)


In [8]:
section_header("5. DATA QUALITY")

# Missing coordinates
no_lat = inv[inv["LATITUD"].isna()]
no_lon = inv[inv["LONGITUD"].isna()]
print(f"Facilities missing LATITUD:  {len(no_lat)}")
print(f"Facilities missing LONGITUD: {len(no_lon)}")

# Missing coordinates by type
if len(no_lat) > 0:
    print("\nMissing coords by facility type:")
    for ftype, n in no_lat["FACILITY_TYPE"].value_counts().items():
        print(f"  {ftype:<40} {n:>5}")

# Edges with missing distances
no_dist = edges[edges["DISTANCE_KM"].isna()]
print(f"\nEdges with missing DISTANCE_KM: {len(no_dist)}")
if len(no_dist) > 0:
    print("  Edge types affected:")
    for et, n in no_dist["EDGE_TYPE"].value_counts().items():
        print(f"    {et:<35} {n:>5}")

# Duplicate edges
dup_cols = ["FROM_NAME","TO_NAME","EDGE_TYPE","COMMODITIES","PRODUCT_FORM"]
dupes = edges[edges.duplicated(subset=dup_cols, keep=False)]
print(f"\nDuplicate edges: {len(dupes)}")
if len(dupes) > 0:
    print(dupes[dup_cols].drop_duplicates().to_string(index=False))

# Integrity label
# port_to_country edges do not carry DISTANCE_KM by design (cross-ocean).
# Exclude them from the missing-distance check.
_no_dist_physical = edges[
    edges["EDGE_TYPE"] != "port_to_country"]["DISTANCE_KM"].isna().sum()
_no_dist_all = edges["DISTANCE_KM"].isna().sum()
_dupes_n = len(edges[edges.duplicated(subset=["FROM_NAME","TO_NAME","EDGE_TYPE","COMMODITIES","PRODUCT_FORM"], keep=False)])
print(f"  (port_to_country edges without distance: {_no_dist_all - _no_dist_physical} — expected, no action needed)")
if _no_dist_physical == 0 and _dupes_n == 0:
    print("\nCHECK 5 (Data quality): PASS")
elif _no_dist_physical < 20 and _dupes_n < 10:
    print(f"\nCHECK 5 (Data quality): WARN  (missing DISTANCE_KM in physical edges: {_no_dist_physical}, duplicate edges: {_dupes_n})")
else:
    print(f"\nCHECK 5 (Data quality): FAIL  (missing DISTANCE_KM in physical edges: {_no_dist_physical}, duplicate edges: {_dupes_n})")



5. DATA QUALITY
Facilities missing LATITUD:  0
Facilities missing LONGITUD: 0

Edges with missing DISTANCE_KM: 1051
  Edge types affected:
    port_to_country                      1051

Duplicate edges: 0
  (port_to_country edges without distance: 1051 — expected, no action needed)

CHECK 5 (Data quality): PASS


In [9]:
section_header("6. EXPORT DESTINATIONS")

if export_df.empty:
    print("  export_df not available in state.")
else:
    print(f"Export records: {len(export_df)}")
    # Export edges use TO_NAME for destination country and COMMODITIES for commodity.
    # The original check for "DESTINATION_COUNTRY" / "COMMODITY" was wrong and
    # would silently skip — fixed to use the actual column names.
    if "TO_NAME" in export_df.columns:
        country_edges = export_df[export_df["EDGE_TYPE"] == "port_to_country"] if "EDGE_TYPE" in export_df.columns else export_df
        print(f"Destination countries: {country_edges['TO_NAME'].nunique()}")
        print("\nTop 15 destinations by export value:")
        val_col = next((c for c in ["EXPORT_VALUE", "FOB_USD"] if c in export_df.columns), None)
        if val_col:
            top = country_edges.groupby("TO_NAME")[val_col].sum().sort_values(ascending=False).head(15)
            for country, val in top.items():
                print(f"  {country:<30} {val:>15,.2f}")
        else:
            print("  (no EXPORT_VALUE column — value totals unavailable)")
    if "COMMODITIES" in export_df.columns:
        print("\nCommodities in exports:")
        for comm, n in export_df["COMMODITIES"].value_counts().items():
            print(f"  {comm:<25} {n:>6} edges")

# Integrity label
if export_df.empty:
    print("\nCHECK 6 (Export destinations): WARN  (export_df is empty — Aduanas data not loaded)")
elif len(export_df) > 10:
    print(f"\nCHECK 6 (Export destinations): PASS  ({len(export_df)} export records)")
else:
    print(f"\nCHECK 6 (Export destinations): WARN  ({len(export_df)} export records — very low)")



6. EXPORT DESTINATIONS
Export records: 1051
Destination countries: 70

Top 15 destinations by export value:
  Switzerland                     872,836,942.09
  USA                             733,699,616.17
  China                           713,567,892.53
  Canada                          342,119,176.17
  Brazil                          186,005,723.92
  Netherlands                     150,366,297.36
  Mexico                          122,905,897.01
  Bahrain                         106,847,182.81
  Spain                            96,850,940.76
  Peru                             86,040,049.06
  South Korea                      75,697,179.45
  South Africa                     53,610,465.97
  Japan                            40,743,873.39
  Ecuador                          33,311,033.18
  Argentina                        21,818,821.63

Commodities in exports:
  Copper                       432 edges
  Nitrate                      147 edges
  Iodine                        84 edges
  Salt  

In [10]:
section_header("7. PORT UTILISATION")

# Only physical Chilean ports — exclude port_to_country edges
port_names = {p['name'] for p in PORTS}
port_edges = edges[
    (edges['TO_NAME'].isin(port_names) | edges['FROM_NAME'].isin(port_names))
    & (edges['EDGE_TYPE'] != 'port_to_country')
]

print(f'Edges touching Chilean ports: {len(port_edges)}')
print('\nEdges per port (arrivals + departures):')
from collections import Counter
port_counts = Counter()
for _, row in port_edges.iterrows():
    if row['TO_NAME'] in port_names:
        port_counts[row['TO_NAME']] += 1
    if row['FROM_NAME'] in port_names:
        port_counts[row['FROM_NAME']] += 1
for port_name, n in sorted(port_counts.items(), key=lambda x: -x[1]):
    print(f'  {port_name:<30} {n:>5} edges')

print(f'\nport_to_country export edges: {len(edges[edges["EDGE_TYPE"] == "port_to_country"])}')
print('(distances not calculated for export destination edges — expected)')

# Integrity label
_n_port_edges = len(port_edges)
_n_ports_used = len(port_counts)
if _n_ports_used >= 8:
    print(f"\nCHECK 7 (Port utilisation): PASS  ({_n_ports_used} ports used, {_n_port_edges} port-touching edges)")
elif _n_ports_used >= 4:
    print(f"\nCHECK 7 (Port utilisation): WARN  ({_n_ports_used} ports used — expected ≥8)")
else:
    print(f"\nCHECK 7 (Port utilisation): FAIL  ({_n_ports_used} ports used — most ports have no edges)")



7. PORT UTILISATION
Edges touching Chilean ports: 90

Edges per port (arrivals + departures):
  Angamos                           18 edges
  Caldera                           11 edges
  Antofagasta (ATI)                  8 edges
  Los Vilos                          8 edges
  Iquique                            8 edges
  San Antonio                        8 edges
  Ventanas                           8 edges
  Coquimbo                           6 edges
  Barquito                           6 edges
  Coloso                             5 edges
  Patache                            2 edges
  Mejillones                         2 edges

port_to_country export edges: 1051
(distances not calculated for export destination edges — expected)

CHECK 7 (Port utilisation): PASS  (12 ports used, 90 port-touching edges)


In [11]:
section_header("8. OUTPUT FILE CHECK")

import glob
expected_files = [
    "Chile_Minerals_Inventory.csv",
    "Chile_Mine_Plant_Links.csv",
    "Chile_Supply_Chain_Edges.csv",
    "Chile_Export_Destinations.csv",
    "Chile_Ports.csv",
    "Mine_Port_Distance_Matrix.csv",
    "Mine_Optimal_Port_Assignments.csv",
]
print(f"Checking {DIR_INTERMED}:")
for fname in expected_files:
    path = os.path.join(DIR_INTERMED, fname)
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        try:
            df = pd.read_csv(path, nrows=1)
            n_cols = len(df.columns)
        except:
            n_cols = "?"
        print(f"  OK    {fname:<45} {size_kb:>8.1f} KB  {n_cols} cols")
    else:
        print(f"  MISSING {fname}")

pkl_files = sorted(glob.glob(os.path.join(DIR_INTERMED, "_pipeline_state_*.pkl")))
print(f"\nState files: {len(pkl_files)}")
for p in pkl_files:
    size_kb = os.path.getsize(p) / 1024
    print(f"  {os.path.basename(p):<35} {size_kb:>8.1f} KB")

# Integrity label
_missing = [f for f in expected_files if not os.path.exists(os.path.join(DIR_INTERMED, f))]
if len(_missing) == 0:
    print(f"\nCHECK 8 (Output files): PASS  (all {len(expected_files)} expected files present)")
elif len(_missing) <= 2:
    print(f"\nCHECK 8 (Output files): WARN  ({len(_missing)} missing: {_missing})")
else:
    print(f"\nCHECK 8 (Output files): FAIL  ({len(_missing)} missing: {_missing})")



8. OUTPUT FILE CHECK
Checking /Users/leoss/Desktop/GitHub/Capstone/Client/chile_analysis/intermediary:
  OK    Chile_Minerals_Inventory.csv                     280.5 KB  163 cols
  OK    Chile_Mine_Plant_Links.csv                       311.5 KB  21 cols
  OK    Chile_Supply_Chain_Edges.csv                     287.0 KB  12 cols
  OK    Chile_Export_Destinations.csv                    128.1 KB  16 cols
  OK    Chile_Ports.csv                                    1.0 KB  6 cols
  OK    Mine_Port_Distance_Matrix.csv                      7.8 KB  14 cols
  OK    Mine_Optimal_Port_Assignments.csv                  1.9 KB  5 cols

State files: 4
  _pipeline_state_0.pkl                  766.5 KB
  _pipeline_state_2.pkl                  791.1 KB
  _pipeline_state_5.pkl                 1101.8 KB
  _pipeline_state_6.pkl                 1002.7 KB

CHECK 8 (Output files): PASS  (all 7 expected files present)


## Save Final State

In [12]:
# Save final state
state.update({
    'inv': inv, 'links': links, 'edges': edges,
    'export_df': export_df, 'ports_df': ports_df,
    'inv_path': inv_path, 'links_path': links_path,
    'cu_total': cu_total, 'n_idle_links': n_idle_links,
})
save_state(state, 6)
print("Final state saved as _pipeline_state_6.pkl")


State saved to /Users/leoss/Desktop/GitHub/Capstone/Client/chile_analysis/intermediary/_pipeline_state_6.pkl
Final state saved as _pipeline_state_6.pkl


---
## Next Step: Generate Visualisations

Run the standalone visualisation script from the project root:

```bash
python3 scripts/chile_visualisations.py
```

Reads `intermediary/_pipeline_state_6.pkl` and writes all charts to `outputs/`.